In [1]:
# Written by Une Butaite & José Carlos A. R.
# Date: 8/June/2025

import cupy as cp
import numpy as np  # needed for uint8 frame buffers passed to the DLL
import ctypes
from PLMController import PLMController
import matplotlib.pyplot as plt

In [2]:
MAX_FRAMES = 120 # Maximum number of RGB frames to be stored in plmctrl's memory space. Each frame contains 24 holograms. -- Limited by your RAM available
N = 904 # PLM width in pixels: 1358
M = 800 # PLM height in pixels: 800

# This is the offset to the PLM virtual monitor. (0, 0) is the top left corner of your main screen.
# The example below ( x0 = 2560, y0 = 0 ) is for the PLM monitor to be on the right of the main QHD screen.
x0 = 2560
y0 = 0

relativePath = r'..\bin\plmctrl.dll'

# Create PLMController instance
plm = PLMController(MAX_FRAMES, N, M, relativePath, x0, y0)

In [3]:
plm.set_windowed(True) # Only for debug purposes -- Suggested if you're testing how this library works.

# set phase map
phase_map = np.array([
    [0, 0, 0, 0],
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [1, 1, 0, 0],
    [0, 0, 1, 0],
    [1, 0, 1, 0],
    [0, 1, 1, 0],
    [1, 1, 1, 0],
    [0, 0, 0, 1],
    [1, 0, 0, 1],
    [0, 1, 0, 1],
    [1, 1, 0, 1],
    [0, 0, 1, 1],
    [1, 0, 1, 1],
    [0, 1, 1, 1],
    [1, 1, 1, 1],
])

phase_map_order = (12, 8, 4, 14, 0, 6, 10, 2, 13, 5, 9, 1, 15, 7, 11, 3)
phase_map = phase_map[phase_map_order,:]
plm.set_phase_map(phase_map)

# Start the UI
plm.start_ui()

In [ ]:
plm.play() # starts reading from the screen in the mode you have configured. plm.stop() to stop the playback

In [ ]:
# Configure the PLM for HDMI. Only run it once.
HDMI = 1
DisplayPort = 2

PlayOnce = 0
Continuous = 1

# Configure the PLM for HDMI
# You should configure the PLM only once per boot. You should also run it
# section by section. Some wait period is necessary between commands.

play_mode = Continuous
connection_type = HDMI

In [ ]:
# Set source to Parallel RGB (0) and port width to 24 bits (1)
plm.set_source(0, 1)

In [ ]:
# Set port swap to Port 0 and 1 to ABC -> ABC
plm.set_port_swap(0, 0)
plm.set_port_swap(1, 0)

In [ ]:
# Set Pixel Mode. 1 for HDMI (Single Pixel), 2 for DisplayPort (Dual Pixel)
plm.set_pixel_mode(connection_type)

In [ ]:
# Set Connection Type (This will lock the PLM to the source (video stream)) -- Wait ~3 sec after this
plm.set_connection_type(connection_type)

In [ ]:
# Set video pattern mode (This is the mode we use for reading from the video stream) -- Wait ~3 sec after this
plm.set_video_pattern_mode()

In [ ]:
# Update LUT with play mode and connection type
plm.update_lut(play_mode, connection_type)

In [ ]:
# Bitpacking and inserting one frame
# Phase is generated on the GPU with CuPy, then transferred to CPU (.get()) before passing to the DLL.

phase = cp.zeros((24, M, N), dtype=cp.float32)
phase[:, :M//4, :N//4] = 0.0
phase[:, :M//4, N//4:] = 0.2
phase[:, M//4:, :N//4] = 0.3
phase[:, M//4:, N//4:] = 0.9

plt.imshow(cp.asnumpy(phase[0, :, :]))

frame = plm.bitpack_holograms_gpu(phase.get())
plm.insert_frames(frame, 0, format=1)

In [4]:
# Bitpacking and inserting one frame at a time
# Phase ramps are computed on the GPU; .get() transfers each frame to CPU for the DLL.

numHolograms = 24

for i in range(MAX_FRAMES):
    a = cp.linspace(0, i * 2 + 1, N, dtype=cp.float32)[None, :]
    b = cp.linspace(0, 0,          M, dtype=cp.float32)[:, None]
    ph = cp.mod(a + b, 1)
    phase = cp.tile(ph[cp.newaxis, :, :], (numHolograms, 1, 1))
    phase = cp.ascontiguousarray(phase)

    plm.bitpack_and_insert_gpu(phase.get(), i)

In [ ]:
# Bitpacking frames first and then inserting them all at once
# Phase is generated on the GPU; the uint8 frame buffer stays on CPU (it goes straight to the DLL).

numHolograms = 24

frames = np.zeros((MAX_FRAMES, *plm.frame_shape), dtype=np.uint8)

for i in range(MAX_FRAMES):
    a = cp.linspace(0, 0,          N, dtype=cp.float32)[None, :]
    b = cp.linspace(0, i * 2 + 1,  M, dtype=cp.float32)[:, None]
    ph = cp.mod(a + b, 1)
    phase = cp.tile(ph[cp.newaxis, :, :], (numHolograms, 1, 1))
    phase_np = cp.ascontiguousarray(phase).get()

    phase_ptr = phase_np.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
    frame_ptr = frames[i].ctypes.data_as(ctypes.POINTER(ctypes.c_uint8))

    plm.bitpack_holograms_gpu_ptr(phase_ptr, frame_ptr, numHolograms)

plm.insert_frames(frames, 0, format=1)

In [4]:
# Create multiple holograms with wedge phases computed on the GPU

x = cp.linspace(-1,     1,    N, dtype=cp.float32)
y = cp.linspace(-M / N, M / N, M, dtype=cp.float32)
xx, yy = cp.meshgrid(x, y)
wedge = lambda alpha, beta: alpha * xx + beta * yy

numHolograms = 24
MAX_FRAMES = 60
phase = cp.zeros((numHolograms, M, N), dtype=cp.float32)

for j in range(MAX_FRAMES):
    print(f"Python: Generating bitpacked hologram #{j + 1}")
    # Uncomment to fill with random wedges:
    for i in range(numHolograms):
        alpha = 50.0 * (cp.random.rand() - 0.5)
        beta  = 50.0 * (cp.random.rand() - 0.5)
        phase[i] = cp.mod(wedge(float(alpha), float(beta)), 2 * cp.pi) / (2 * cp.pi)

    plm.bitpack_and_insert_gpu(phase.get(), j)

Python: Generating bitpacked hologram #1
Python: Generating bitpacked hologram #2
Python: Generating bitpacked hologram #3
Python: Generating bitpacked hologram #4
Python: Generating bitpacked hologram #5
Python: Generating bitpacked hologram #6
Python: Generating bitpacked hologram #7
Python: Generating bitpacked hologram #8
Python: Generating bitpacked hologram #9
Python: Generating bitpacked hologram #10
Python: Generating bitpacked hologram #11
Python: Generating bitpacked hologram #12
Python: Generating bitpacked hologram #13
Python: Generating bitpacked hologram #14
Python: Generating bitpacked hologram #15
Python: Generating bitpacked hologram #16
Python: Generating bitpacked hologram #17
Python: Generating bitpacked hologram #18
Python: Generating bitpacked hologram #19
Python: Generating bitpacked hologram #20
Python: Generating bitpacked hologram #21
Python: Generating bitpacked hologram #22
Python: Generating bitpacked hologram #23
Python: Generating bitpacked hologram #24
P

In [5]:
plm.cleanup()

In [22]:
plm.start_sequence(100)